### 라이브러리 불러오기

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder


In [3]:
users = pd.read_csv('C:/Users/USER/Desktop/data/cleaned/users_EDA.csv')
events = pd.read_csv('C:/Users/USER/Desktop/data/cleaned/events_EDA.csv')
orders = pd.read_csv('C:/Users/USER/Desktop/data/cleaned/orders_EDA.csv')

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11916\1429863484.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  users = pd.read_csv('C:/Users/USER/Desktop/data/cleaned/users_EDA.csv')


In [4]:
# 2. 주문 횟수 집계 및 3회 이상은 3으로 통합
order_counts = orders.groupby('user_id').size().reset_index(name='order_count')
order_counts['order_count'] = order_counts['order_count'].apply(lambda x: x if x <= 3 else 3)


In [5]:
# 3. users와 주문 횟수 병합 (없으면 0)
users = users.merge(order_counts, left_on='id', right_on='user_id', how='left')
users['order_count'] = users['order_count'].fillna(0).astype(int)


In [6]:
# 4. 가입일(created_at)과 첫 이벤트일(created_at_first_event) 차이 (가입 이후 바로 활동 유무)
first_event = events.groupby('user_id')['created_at'].min().reset_index()
first_event.rename(columns={'created_at': 'created_at_first_event'}, inplace=True)
users = users.merge(
    first_event[['user_id', 'created_at_first_event']],
    left_on='id', right_on='user_id',
    how='left',
    suffixes=('', '_drop')
)
if 'user_id_drop' in users.columns:
    users.drop(columns=['user_id_drop'], inplace=True)

In [7]:
# 날짜 컬럼을 datetime 타입으로 변환
users['created_at'] = pd.to_datetime(users['created_at'], errors='coerce')
users['created_at_first_event'] = pd.to_datetime(users['created_at_first_event'], errors='coerce')

In [8]:
# 가입일~첫 이벤트까지 걸린 일수 (결측 시 -1)
users['days_to_first_event'] = (users['created_at_first_event'] - users['created_at']).dt.days
users['days_to_first_event'] = users['days_to_first_event'].fillna(-1).astype(int)


In [9]:
# 5. 카테고리 변수 인코딩 (gender, traffic_source)
cat_cols = ['gender', 'traffic_source']
for col in cat_cols:
    users[col] = users[col].astype(str)  # 결측 방지용 str 변환
    le = LabelEncoder()
    users[col] = le.fit_transform(users[col])

In [10]:
# 6. 모델에 쓸 피처 정의
feature_cols = ['age', 'gender', 'traffic_source', 'days_to_first_event']

In [11]:
X = users[feature_cols].fillna(-1)  # 결측은 -1로 대체
y = users['order_count']


In [12]:
# 7. train/test split (계층적 샘플링)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [13]:
# 8. LightGBM 데이터셋 변환
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_test, label=y_test)

In [14]:
# 9. LightGBM 파라미터 (다중 클래스 분류)
params = {
    'objective': 'multiclass',
    'num_class': 4,  # 0,1,2,3으로 4개 클래스
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'seed': 42,
    'class_weight': 'balanced',  # 클래스 불균형 대응
}

In [15]:
import lightgbm as lgb
print(lgb.__version__)

4.6.0


In [16]:
model = lgb.train(
    params,
    train_data,
    valid_sets=[valid_data],
    num_boost_round=500,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

Training until validation scores don't improve for 50 rounds
[50]	valid_0's multi_logloss: 0.67947
[100]	valid_0's multi_logloss: 0.678444
Early stopping, best iteration is:
[76]	valid_0's multi_logloss: 0.6778


In [17]:
# 11. 예측 및 평가
y_pred_probs = model.predict(X_test)  # 각 클래스별 확률
y_pred = y_pred_probs.argmax(axis=1)

In [18]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.70505
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4373
           1       0.62      1.00      0.77      9732
           2       0.35      0.01      0.02      3954
           3       0.00      0.00      0.00      1941

    accuracy                           0.71     20000
   macro avg       0.49      0.50      0.45     20000
weighted avg       0.59      0.71      0.60     20000



In [19]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import optuna

# X, y는 전체 데이터와 라벨
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def objective(trial):
    param = {
        'objective': 'multiclass',
        'num_class': 4,
        'metric': 'multi_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'seed': 42,
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'lambda_l1': trial.suggest_float('lambda_l1', 0, 5),
        'lambda_l2': trial.suggest_float('lambda_l2', 0, 5),
    }

    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_valid, label=y_valid)

    model = lgb.train(
        param,
        train_data,
        valid_sets=[valid_data],
        num_boost_round=1000,
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=0),
        ],
    )

    preds = model.predict(X_valid, num_iteration=model.best_iteration)
    pred_labels = preds.argmax(axis=1)
    accuracy = accuracy_score(y_valid, pred_labels)
    return accuracy

import optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("Best trial:")
trial = study.best_trial
print(f"  Accuracy: {trial.value}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

# 최적 파라미터로 최종 모델 학습 (원하는 경우)
best_params = trial.params
best_params.update({
    'objective': 'multiclass',
    'num_class': 4,
    'metric': 'multi_logloss',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'seed': 42,
})

final_train = lgb.Dataset(X, label=y)
final_model = lgb.train(best_params, final_train, num_boost_round=study.best_trial.user_attrs.get('best_iteration', 100))


[I 2025-08-11 10:44:30,898] A new study created in memory with name: no-name-3ca709bf-420b-430b-9008-c29d053a4e58


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[441]	valid_0's multi_logloss: 0.680738


[I 2025-08-11 10:45:08,962] Trial 0 finished with value: 0.70405 and parameters: {'num_leaves': 137, 'max_depth': 16, 'learning_rate': 0.04065989952238642, 'min_data_in_leaf': 80, 'feature_fraction': 0.5923943823118225, 'bagging_fraction': 0.7305313304617934, 'bagging_freq': 5, 'lambda_l1': 3.4174550701870965, 'lambda_l2': 1.4124829409886197}. Best is trial 0 with value: 0.70405.


Training until validation scores don't improve for 50 rounds


[I 2025-08-11 10:45:13,155] Trial 1 finished with value: 0.70485 and parameters: {'num_leaves': 113, 'max_depth': 5, 'learning_rate': 0.11346950799658251, 'min_data_in_leaf': 69, 'feature_fraction': 0.918975643866805, 'bagging_fraction': 0.5296419342294876, 'bagging_freq': 5, 'lambda_l1': 3.954017276343855, 'lambda_l2': 2.8651242113956306}. Best is trial 1 with value: 0.70485.


...

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[144]	valid_0's multi_logloss: 0.679288


[I 2025-08-11 11:19:58,934] Trial 48 finished with value: 0.7042 and parameters: {'num_leaves': 78, 'max_depth': 6, 'learning_rate': 0.16815825169524412, 'min_data_in_leaf': 67, 'feature_fraction': 0.5828054203662112, 'bagging_fraction': 0.7387484650463454, 'bagging_freq': 4, 'lambda_l1': 0.39923969228609957, 'lambda_l2': 3.52453205538681}. Best is trial 4 with value: 0.70525.


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's multi_logloss: 0.710233


[I 2025-08-11 11:20:42,318] Trial 49 finished with value: 0.70525 and parameters: {'num_leaves': 24, 'max_depth': 19, 'learning_rate': 0.0021795365384075496, 'min_data_in_leaf': 41, 'feature_fraction': 0.8977339984148823, 'bagging_fraction': 0.6827862914267399, 'bagging_freq': 3, 'lambda_l1': 1.3214040816175059, 'lambda_l2': 4.78963766597086}. Best is trial 4 with value: 0.70525.


Best trial:
  Accuracy: 0.70525
  Params: 
    num_leaves: 22
    max_depth: 6
    learning_rate: 0.0051061087876178215
    min_data_in_leaf: 67
    feature_fraction: 0.965073314886395
    bagging_fraction: 0.9793459024122727
    bagging_freq: 9
    lambda_l1: 4.117776561594546
    lambda_l2: 2.6394234704838655


- num_leaves : 트리 하나가 가질 수 있는 최대 잎사귀의 수, 모델 복잡도와 표현력을 조절
- max_depth : 트리의 최대 길이로, 너무 깊으면 과적합, 너무 얇으면 학습 부족
- learning_rate : 학습률, 모델이 한번에 얼마나 많이 가중치를 업데이트할지 결정
- min_date_in_leaf : 하나의 잎사귀가 갖춰야 할 최소 데이터 수 너무 작으면 과적합
- feature_fraction : 각 트리를 만들 떄 임의로 선택하는 피처의 비율로, 과적합 방지 및 학습 속도 향상에 도움을 줌
- bagging_fraction : 데이터 샘플을 임의로 선택해 학습에 사용하는 비율, 배깅 효과로 모델 다양성을 키워 성능 향상에 기여
- bagging_freq : 샘플링을 수행하는 빈도, 몇 번마다 한 번씩 쌤플링을 할지 정하는 값
- lambda_L1 : L1 정규화 강도, 모델 가중치의 절대값 합에 패널티를 주어 과적합을 줄이고 중요하지 않은 피처를 희석
- lambda_L2 : L2정규화 강도이며, 모델 가중치의 제곱 합에 패널티를 줘서 가중치가 너무 커지는 걸 막고 안정화시킴